# Check KinematiK yourself, in your browser

This notebook re-runs KinematiK's published checks on Google's machines, with nothing to install on yours. Run the
cells in order (Runtime → Run all). It takes about 10 minutes, most of it installing.

Each cell prints a pass/fail summary. If anything fails, or a number doesn't match what README.md or VALIDATION.md
claims, that's a break: report it with the **I reproduced it / I couldn't** issue template, and it is credited in
[CHALLENGE.md](https://github.com/frederikexd/kinematik/blob/main/CHALLENGE.md) either way.

In [ ]:
# 1. Get the code (latest main) and install it
!git clone --depth 1 https://github.com/frederikexd/kinematik.git
%cd kinematik
!pip install -q -r requirements.txt pytest pytest-xdist hypothesis
!pip install -q -e ./fsae_suspension
!git log -1 --format='Checking commit %h (%cd)'

## 2. Powertrain benchmarks
NREL FASTSim, five published GT-SUITE studies, the blind Simscape cases, measured data, and the garbage-in
benchmark. These are the tests behind the powertrain tables in README.md.

In [ ]:
%cd /content/kinematik/fsae_suspension
!python -m pytest -q -n auto tests/test_fastsim_benchmark.py tests/test_gt_suite_benchmark.py tests/test_powertrain_benchmark.py tests/test_gigo_benchmark.py tests/test_gigo_round3.py tests/test_plausibility.py

## 3. Suspension benchmarks
Adams View, Lotus, and the property-based invariants behind the suspension table in README.md.

In [ ]:
!python -m pytest -q -n auto tests/test_adams_view_benchmark.py tests/test_lotus_benchmark.py tests/test_invariants_property.py

## 4. The prediction ledger: real cars, official results
Every sealed record is checked: unchanged since it was sealed, made by the same code, and still reproduced by
today's code. Then the registered tests are re-run from scratch and their numbers printed, so you can compare them
with docs/PREDICTION_LEDGER.md.

In [ ]:
%cd /content/kinematik
import glob, subprocess
for rec in sorted(glob.glob('benchmarks/ledger/fsg*/SEALED*.json')):
    script = 'ledger_v2.py' if 'v2' in rec else 'ledger.py'
    out = subprocess.run(['python', f'fsae_suspension/scripts/{script}', 'verify', '--record', rec],
                         capture_output=True, text=True)
    print(rec, out.stdout.strip() or out.stderr.strip())
!python fsae_suspension/scripts/ledger_v2.py registered | head -60

## 5. Optional: re-solve a published suspension corner (about 4 minutes)
Re-runs the final rear corner and requires every coordinate to match the recorded output to 1e-6 mm.

In [ ]:
%cd /content/kinematik/fsae_suspension
!python scripts/verify_run.py manifests/rear_v11_final.recorded.genesis.json